# LeetCode #52: N-Queens II

https://leetcode.com/problems/n-queens-ii/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n^n)$ | $O(n)$ |
| **Optimal: Backtracking with Bitmask ★** | $O(n!)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force
Try all $n^n$ column assignments (one per row) and count those with no attacking pairs. Extremely slow because no pruning happens — invalid partial states are extended until all $n$ rows are filled.

### Optimal: Backtracking with Bitmask ★
Same bitmask strategy as N-Queens but only counts solutions, skipping board reconstruction entirely. Three integer masks track occupied columns and the two diagonal families. Only columns free of all three masks are tried at each row, pruning the search tree to $O(n!)$ nodes. The counter is incremented whenever all $n$ rows are successfully placed.

**Constraints:**
* 1 <= n <= 9

## Solutions
### C#

In [ ]:
// Backtracking with bitmask: count solutions without building boards
public class Solution {
    public int TotalNQueens(int n) {
        // full = bitmask with n lowest bits set (all columns valid at the start)
        return Backtrack(0, 0, 0, n, (1 << n) - 1);
    }

    private int Backtrack(int colMask, int diagMask, int antiMask, int n, int full) {
        if (colMask == full) {
            // All n columns placed — one valid arrangement found
            return 1;
        }
        int count = 0;
        // Available columns: bits not blocked by any of the three attack masks
        int available = full & ~(colMask | diagMask | antiMask);
        while (available != 0) {
            // Pick the lowest free column bit
            int bit = available & (-available);
            available -= bit;
            // Propagate diagonal influence one row down before recursing
            count += Backtrack(
                colMask | bit,
                (diagMask | bit) >> 1,
                (antiMask | bit) << 1,
                n, full);
        }
        return count;
    }
}

### Python

In [ ]:
# Backtracking with bitmask: count solutions without building boards
class Solution:
    def totalNQueens(self, n: int) -> int:
        full = (1 << n) - 1  # all n column bits set — fully occupied mask

        def backtrack(col_mask, diag_mask, anti_mask):
            if col_mask == full:
                # All n queens placed — count this arrangement
                return 1
            count = 0
            # Available columns: bits not blocked by any attack mask
            available = full & ~(col_mask | diag_mask | anti_mask)
            while available:
                # Isolate rightmost free column
                bit = available & (-available)
                available -= bit
                # Advance diagonal masks by one row before recursing
                count += backtrack(
                    col_mask | bit,
                    (diag_mask | bit) >> 1,
                    (anti_mask | bit) << 1)
            return count

        return backtrack(0, 0, 0)

### Go

In [ ]:
// Backtracking with bitmask: count solutions without building boards
package main

func totalNQueens(n int) int {
    full := (1 << n) - 1 // all n column bits set

    var backtrack func(colMask, diagMask, antiMask int) int
    backtrack = func(colMask, diagMask, antiMask int) int {
        if colMask == full {
            // All n queens placed — this is a valid arrangement
            return 1
        }
        count := 0
        // Available columns: bits not blocked by any attack mask
        available := full & ^(colMask | diagMask | antiMask)
        for available != 0 {
            // Pick the lowest free column bit
            bit := available & (-available)
            available -= bit
            // Shift diagonal masks as influence propagates to the next row
            count += backtrack(
                colMask|bit,
                (diagMask|bit)>>1,
                (antiMask|bit)<<1)
        }
        return count
    }

    return backtrack(0, 0, 0)
}

### Rust

In [ ]:
// Backtracking with bitmask: count solutions without building boards
impl Solution {
    pub fn total_n_queens(n: i32) -> i32 {
        let full = (1usize << n) - 1; // all n column bits set
        Self::backtrack(0, 0, 0, full) as i32
    }

    fn backtrack(col_mask: usize, diag_mask: usize, anti_mask: usize, full: usize) -> usize {
        if col_mask == full {
            // All n queens placed — count this valid arrangement
            return 1;
        }
        let mut count = 0;
        // Available columns: bits not blocked by any attack mask
        let mut available = full & !(col_mask | diag_mask | anti_mask);
        while available != 0 {
            // Isolate lowest set bit (rightmost free column)
            let bit = available & available.wrapping_neg();
            available -= bit;
            // Advance diagonal masks by one row before recursing
            count += Self::backtrack(
                col_mask | bit,
                (diag_mask | bit) >> 1,
                (anti_mask | bit) << 1,
                full,
            );
        }
        count
    }
}

## Example Scenarios

**1. Common Case** — $n = 4$

**Input:** `n = 4`
Two valid arrangements exist. The bitmask prunes most paths early — after placing the first queen, entire columns and diagonals are masked off, reducing the second row's choices from 4 to at most 2.

**2. Slightly Complex** — $n = 6$

**Input:** `n = 6`
There are 4 distinct solutions. No board strings are built — only a running counter is maintained, making this faster than N-Queens I for the same input size.

**3. Edge Case: Time Factor** — $n = 9$

**Input:** `n = 9`
Largest allowed input with 352 solutions. The bitmask prunes the search to $O(9!)$ paths rather than $9^9$, keeping runtime comfortably under a millisecond.

**4. Edge Case: Space Factor** — $n = 1$

**Input:** `n = 1`
Trivial case — one queen in a $1 \times 1$ board. The recursion immediately hits `col_mask == full` and returns 1. Stack depth is 1, space is $O(1)$.

**5. Almost-Impossible but Plausible** — $n = 8$ with manually traced mask values

**Input:** `n = 8`
92 solutions. After placing the first queen in column 0, `col_mask = 0b00000001`, `diag_mask = 0b00000001 >> 1 = 0`, `anti_mask = 0b00000010`. In row 1 only 5 of 8 columns remain available — the bitmask intersection eliminates columns 0, 1, and 7 at once with a single bitwise OR.